# Milestone 5 Lab, notebook 2 of 7: A loader that is safe to repeat

## Where we are

| Notebook | What you do |
|---|---|
| 1. Files arrive, by hand | Load the files the naive way, and watch three things go wrong |
| **2. A loader that is safe to repeat** (you are here) | Fingerprint each file, check its columns, and choose a rule for every file |
| 3. AI reads a changed file | Ask a model what a renamed column means, and score its suggestions |
| 4. A workflow that can wait | Build the loader as a graph that pauses for your approval |
| 5. Crash and carry on | Restart the kernel on purpose, and find the file still waiting |
| 6. A swarm of file workers | Load fifty files side by side, and score the run for quality, cost, and time |
| 7. The same question, from loaded files | Approve the waiting files, run the pipeline, and ask the leader's question |

Model requests in this notebook: **0**. A full run of this lab costs about 3 US cents.

## What this notebook is for

The naive loader had two weaknesses: it could not tell that it had seen a file before, and it never compared a file's columns with what it expected.

Here you fix both, with plain code and no AI. By the end, every one of the fifty files has a **rule** that says what happens to it.

In [ ]:
import pandas as pd
from lab_helpers import LAB, LANDING, fingerprint_by_hand, reset_lab, show_doc

reset_lab()

`reset_lab()` clears the lab's own folders, so you can run this notebook again from a clean start. The lab never changes the landing files.

## Step 1: The rules the business agreed

In [ ]:
show_doc("HLD.md", "5. The rules for a file")

## Step 2: Recognise a file by its content

In milestone 4 you built a **fingerprint**: a short code worked out from a file's content. Equal fingerprints mean equal content. That is exactly what we need.

In [ ]:
original = LANDING / "orders_2026-09-14.csv"
resent = LANDING / "orders_2026-09-14_resend.csv"

print(original.name, fingerprint_by_hand(original))
print(resent.name, fingerprint_by_hand(resent))

Different names, the same fingerprint. Now a loader that remembers what it has loaded. It keeps a dictionary: fingerprint, and the file that loaded it.

In [ ]:
already_loaded = {}


def load_once(file):
    """Load a file, unless a file with the same content was loaded before."""
    fingerprint = fingerprint_by_hand(file)
    if fingerprint in already_loaded:
        return f"skipped: same content as {already_loaded[fingerprint]}"
    already_loaded[fingerprint] = file.name
    return f"loaded {len(pd.read_csv(file))} rows"


print(original.name, "->", load_once(original))
print(resent.name, "->", load_once(resent))
print(original.name, "->", load_once(original))

The third line matters as much as the second. Loading the **same file again** also does nothing. A loader with this property is called **idempotent**: doing it twice gives the same result as doing it once. That is what makes it safe to run a load again after something went wrong.

## Step 3: Compare a file's columns with what we expect

Milestone 2 wrote down the columns of every source table, as a contract. The loader reads them from there.

In [ ]:
from retail_profit_stack.ingest.feeds import expected_columns

expected = expected_columns("order_items")
expected

Now one of the changed files. We only need its first line.

In [ ]:
file = LANDING / "order_items_2026-10-12.csv"

columns = file.read_text(encoding="utf-8").splitlines()[0].split(",")

missing = [column for column in expected if column not in columns]
unknown = [column for column in columns if column not in expected]

print("The file's columns:", columns)
print("Missing:", missing)
print("Unknown:", unknown)

Two lists, and they are all the rule needs:

- **Missing:** columns we expect that the file does not have.
- **Unknown:** columns the file has that we do not expect.

## Step 4: The rule, by hand

Here is the choice of rule as a small function. Read it against the table in step 1. (Rules 1 and 5 need the loader's memory, and come in a moment.)

In [ ]:
def rule_for(missing, unknown):
    """The rule for a file that was not loaded before and has no approved mapping."""
    if not missing and not unknown:
        return 2, "load"
    if not missing:
        return 3, "load"            # only extra columns, which are ignored
    if not unknown:
        return 4, "rejected"        # something is missing, and nothing new stands in for it
    return 6, "needs_mapping"       # perhaps a rename: ask


print(rule_for(missing, unknown))

Apply it to every file in the landing folder.

In [ ]:
from retail_profit_stack.ingest.feeds import feed_of

rows = []
for file in sorted(LANDING.glob("*.csv")):
    expected = expected_columns(feed_of(file.name))
    columns = file.read_text(encoding="utf-8").splitlines()[0].split(",")
    missing = [column for column in expected if column not in columns]
    unknown = [column for column in columns if column not in expected]
    rule, outcome = rule_for(missing, unknown)
    rows.append({"file": file.name, "rule": rule, "outcome": outcome, "missing": missing, "unknown": unknown})

by_hand = pd.DataFrame(rows)
by_hand["outcome"].value_counts()

Most files are fine. Here are the ones that are not plain rule 2.

In [ ]:
pd.set_option("display.max_colwidth", 60)
by_hand[by_hand["rule"] != 2]

Check it against notebook 1. The four shipment files with `warehouse_code` are rule 3: loaded, with the extra column ignored. The incomplete refund file is rule 4: rejected. Six files need a mapping. The file with reordered columns is not in this list at all, because columns are matched by name.

One file is still wrong: the resent orders file shows as an ordinary rule 2. Rule 1 needs the loader's memory.

## Step 5: The production rule function

`inspect_file` in production is your `rule_for` with rule 1 and rule 5 added. It takes the loader's memory as arguments: the fingerprints already loaded, and the mappings a person has approved.

In [ ]:
from retail_profit_stack.ingest.rules import inspect_file

found = inspect_file(resent, expected_columns("orders"), loaded={}, approved=[])
print("Nothing loaded yet:      rule", found.rule, "-", found.outcome)

loaded = {fingerprint_by_hand(original): original.name}
found = inspect_file(resent, expected_columns("orders"), loaded=loaded, approved=[])
print("After the original loads: rule", found.rule, "-", found.outcome)
print(found.reason)

And rule 5. Suppose a person has already approved what `list_price` and `discount` mean. A statement like that is called a **mapping**.

In [ ]:
mapping = {"list_price": "unit_list_price", "discount": "discount_amount"}
file = LANDING / "order_items_2026-10-12.csv"

found = inspect_file(file, expected_columns("order_items"), loaded={}, approved=[])
print("No approved mapping:  rule", found.rule, "-", found.outcome)

found = inspect_file(file, expected_columns("order_items"), loaded={}, approved=[mapping])
print("An approved mapping:  rule", found.rule, "-", found.outcome)

Who decides that mapping is the subject of the next two notebooks.

## Step 6: The load record

The dictionary in step 2 lives in this notebook's memory. Restart the kernel and it is gone, and the loader would load everything again.

So the loader's memory goes in a small database file, using **SQLite**: a database that is a single file on your computer, with nothing to install. The **load record** has a table whose key is the fingerprint.

In [ ]:
from retail_profit_stack.ingest import ledger

print(ledger.CREATE_TABLES)

`PRIMARY KEY` on `fingerprint` means the database itself refuses a second row with the same fingerprint. `claim_content` asks the database to add a fingerprint and to **do nothing if it is already there**, then returns the file that owns it.

In [ ]:
fingerprint = fingerprint_by_hand(original)

print(ledger.claim_content(LAB.state_file, fingerprint, original.name))
print(ledger.claim_content(LAB.state_file, fingerprint, resent.name))
print(ledger.claim_content(LAB.state_file, fingerprint, original.name))

All three calls name the original file as the owner. The first call made it the owner. The second shows the resent file that it lost: its content was already loaded. The third shows that asking again is harmless.

Because the database does this in one step, it works even when two copies of a file are loaded at the same moment. That will matter in notebook 6.

In [ ]:
ledger.loaded_fingerprints(LAB.state_file)

## What we now know

- A file is recognised by the **fingerprint of its content**. A new name does not fool it.
- An **idempotent** loader gives the same result whether it runs once or twice. That makes a rerun safe.
- Two lists decide a file's rule: the columns that are **missing**, and the columns that are **unknown**.
- Code applies one of six rules. 42 of the 50 files simply load. One is skipped, one is rejected.
- Six files have a renamed column. Code can see that something changed. It cannot know what the new names **mean**.

**Next:** notebook 3. Working out what a renamed column means is reading and judgement. That is a job for AI, with a person checking.